# Resume Screening and Skill Matching System
This notebook demonstrates the end-to-end pipeline covering:
1. Data Ingestion
2. Text Preprocessing & NLP
3. Embedding Generation via Sentence Transformers
4. Candidate Ranking via Cosine Similarity
5. Metrics Evaluation

In [ ]:
import os
import sys
sys.path.append(os.path.abspath('..'))
from src.ingestion import load_documents_from_directory
from src.preprocessing import preprocess, extract_skills
from src.embedding import generate_embedding, generate_embeddings_batch
from src.ranking import rank_candidates, evaluate_metrics

In [ ]:
# 1. Ingestion
resumes_dir = '../data/resumes'
jd_dir = '../data/job_descriptions'

resumes = load_documents_from_directory(resumes_dir)
jds = load_documents_from_directory(jd_dir)

print(f"Loaded {len(resumes)} resumes and {len(jds)} job descriptions.")
jd_filename, jd_text = list(jds.items())[0]

In [ ]:
# 2. Preprocessing
jd_processed = preprocess(jd_text)
jd_skills = extract_skills(jd_text)

processed_resumes = {}
resume_skills = {}
for fname, text in resumes.items():
    processed_resumes[fname] = preprocess(text)
    resume_skills[fname] = extract_skills(text)
print("JD Skills:", jd_skills)

In [ ]:
# 3. Embeddings Generation
jd_embedding = generate_embedding(jd_processed)
resume_filenames = list(processed_resumes.keys())
resume_texts = [processed_resumes[fname] for fname in resume_filenames]
resume_embeddings = generate_embeddings_batch(resume_texts)
print("Generated embeddings successfully.")

In [ ]:
# 4. Similarity and Ranking
ranked_candidates = rank_candidates(jd_embedding, resume_embeddings, resume_filenames)
print("\nRanked Candidates for:", jd_filename)
for rank, (fname, score) in enumerate(ranked_candidates, 1):
    print(f"{rank}. {fname} - Score: {score:.4f} - Skills: {resume_skills[fname]}")

In [ ]:
# 5. Evaluation
# Assuming 'resume_a.txt' is the ground truth for this python developer JD
ground_truth = {'resume_a.txt'}
ranked_filenames = [item[0] for item in ranked_candidates]
metrics = evaluate_metrics(ranked_filenames, ground_truth)

print("\nEvaluation Metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")